In [48]:
# Optional config for better memory efficiency
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Required imports
import torch
from mapanything.models import MapAnything
from mapanything.utils.image import load_images

import open3d as o3d
import numpy as np

# Get inference device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Init model - This requires internet access or the huggingface hub cache to be pre-downloaded
# For Apache 2.0 license model, use "facebook/map-anything-apache"
model = MapAnything.from_pretrained("facebook/map-anything").to(device)

# Load and preprocess images from a folder or list of paths

images = ["/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output106044.png"] # or ["path/to/img1.jpg", "path/to/img2.jpg", ...]
   # Denormalized input images for visualization (B, H, W, 3)

views = load_images(images)

# Run inference
predictions = model.infer(
    views,                            # Input views
    memory_efficient_inference=False, # Trades off speed for more views (up to 2000 views on 140 GB)
    use_amp=True,                     # Use mixed precision inference (recommended)
    amp_dtype="bf16",                 # bf16 inference (recommended; falls back to fp16 if bf16 not supported)
    apply_mask=True,                  # Apply masking to dense geometry outputs
    mask_edges=True,                  # Remove edge artifacts by using normals and depth
    apply_confidence_mask=False,      # Filter low-confidence regions
    confidence_percentile=10,         # Remove bottom 10 percentile confidence pixels
)

# Access results for each view - Complete list of metric outputs
for i, pred in enumerate(predictions):
    # Geometry outputs
    pts3d = pred["pts3d"]                     # 3D points in world coordinates (B, H, W, 3)
    pts3d_cam = pred["pts3d_cam"]             # 3D points in camera coordinates (B, H, W, 3)
    depth_z = pred["depth_z"]                 # Z-depth in camera frame (B, H, W, 1)
    depth_along_ray = pred["depth_along_ray"] # Depth along ray in camera frame (B, H, W, 1)

    # Camera outputs
    ray_directions = pred["ray_directions"]   # Ray directions in camera frame (B, H, W, 3)
    intrinsics = pred["intrinsics"]           # Recovered pinhole camera intrinsics (B, 3, 3)
    camera_poses = pred["camera_poses"]       # OpenCV (+X - Right, +Y - Down, +Z - Forward) cam2world poses in world frame (B, 4, 4)
    cam_trans = pred["cam_trans"]             # OpenCV (+X - Right, +Y - Down, +Z - Forward) cam2world translation in world frame (B, 3)
    cam_quats = pred["cam_quats"]             # OpenCV (+X - Right, +Y - Down, +Z - Forward) cam2world quaternion in world frame (B, 4)

    # Quality and masking
    confidence = pred["conf"]                 # Per-pixel confidence scores (B, H, W)
    mask = pred["mask"]                       # Combined validity mask (B, H, W, 1)
    non_ambiguous_mask = pred["non_ambiguous_mask"]                # Non-ambiguous regions (B, H, W)
    non_ambiguous_mask_logits = pred["non_ambiguous_mask_logits"]  # Mask logits (B, H, W)

    # Scaling
    metric_scaling_factor = pred["metric_scaling_factor"]  # Applied metric scaling (B,)

    # Original input
    img_no_norm = pred["img_no_norm"]      


# --- Place this code after you have the 'predictions' variable ---

# Get the result for the first image
pred = predictions[0]

# 1. Get the 3D points and reshape them
points = pred["pts3d"].squeeze().cpu().numpy().reshape(-1, 3)

# 2. Get the color data and reshape it
colors = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)



# 3. Normalize colors for Open3D
# Open3D expects colors as floats between 0.0 and 1.0, not 0-255.


# 4. Create the point cloud object
pcd = o3d.geometry.PointCloud()

# 5. Assign both points AND colors
pcd.points = o3d.utility.Vector3dVector(points)
pcd.colors = o3d.utility.Vector3dVector(colors)

coordinate_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(
    size=1.0, origin=[0, 0, 0]
)

transform_matrix = np.array([
    [1,  0,  0,  0],  
    [0, -1,  0,  0],  
    [0,  0, -1,  0],  
    [0,  0,  0,  1]
])

pcd.transform(transform_matrix)
coordinate_frame.transform(transform_matrix)
# 6. Display the now colored point cloud
print("Displaying colored point cloud...")
o3d.visualization.draw_geometries([pcd,coordinate_frame])

Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /home/tong/.cache/torch/hub/facebookresearch_dinov2_main


Displaying colored point cloud...


In [9]:
# (Your initial imports and model setup code remains the same)
# ...

# 1. Provide a list of multiple image paths
images = [
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output104520.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output104580.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output104640.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output104700.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output104760.png",

]

views = load_images(images)

# Run inference (this will process all images in the list)
predictions = model.infer(
    views,                            # Input views
    memory_efficient_inference=False, # Trades off speed for more views (up to 2000 views on 140 GB)
    use_amp=True,                     # Use mixed precision inference (recommended)
    amp_dtype="bf16",                 # bf16 inference (recommended; falls back to fp16 if bf16 not supported)
    apply_mask=True,                  # Apply masking to dense geometry outputs
    mask_edges=True,                  # Remove edge artifacts by using normals and depth
    apply_confidence_mask=False,      # Filter low-confidence regions
    confidence_percentile=10,         # Remove bottom 10 percentile confidence pixels
)





geometries_to_draw = []
camera_positions = []


world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0, origin=[0, 0, 0])
geometries_to_draw.append(world_frame)


for pred in predictions:
    
    points_cam = pred["pts3d_cam"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    colors = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points_cam)
    pcd.colors = o3d.utility.Vector3dVector(colors)
    
    
    camera_pose = pred["camera_poses"].squeeze().cpu().numpy()
    
    
    pcd.transform(camera_pose)

    camera_center = camera_pose[:3, 3]
    camera_positions.append(camera_center)
    
    
    camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame.transform(camera_pose)
    
    
    geometries_to_draw.append(pcd)
    geometries_to_draw.append(camera_frame)

if len(camera_positions) > 1:
    # Define the points for the line set
    line_points = o3d.utility.Vector3dVector(camera_positions)
    # Define which points to connect (0->1, 1->2, etc.)
    line_indices = [[i, i + 1] for i in range(len(camera_positions) - 1)]
    lines = o3d.utility.Vector2iVector(line_indices)
    
    # Create the LineSet object
    camera_path = o3d.geometry.LineSet(points=line_points, lines=lines)
    
    # Set the color of the path to red
    camera_path.paint_uniform_color([1, 0, 0])
    
    # Add the path to our list of things to draw
    geometries_to_draw.append(camera_path)

transform_matrix = np.array([
    [1,  0,  0,  0],  # Stays the same
    [0, -1,  0,  0],  # Inverts Y
    [0,  0, -1,  0],  # Inverts Z
    [0,  0,  0,  1]
])

for geometry in geometries_to_draw:
    geometry.transform(transform_matrix)
# 4. Display all the geometries together in one window
print(f"Displaying combined scene with {len(predictions)} point clouds...")
o3d.visualization.draw_geometries(geometries_to_draw)

Displaying combined scene with 5 point clouds...


In [8]:
# (Your initial imports and model setup code remains the same)
# ...

# 1. Provide a list of multiple image paths
images = [
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output104520.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output104778.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output105058.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output105310.png",
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/output106379.png",

]

views = load_images(images)

# Run inference (this will process all images in the list)
predictions = model.infer(
    views,                            # Input views
    memory_efficient_inference=False, # Trades off speed for more views (up to 2000 views on 140 GB)
    use_amp=True,                     # Use mixed precision inference (recommended)
    amp_dtype="bf16",                 # bf16 inference (recommended; falls back to fp16 if bf16 not supported)
    apply_mask=True,                  # Apply masking to dense geometry outputs
    mask_edges=True,                  # Remove edge artifacts by using normals and depth
    apply_confidence_mask=False,      # Filter low-confidence regions
    confidence_percentile=10,         # Remove bottom 10 percentile confidence pixels
)





geometries_to_draw = []
camera_positions = []


world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0, origin=[0, 0, 0])
geometries_to_draw.append(world_frame)


for pred in predictions:
    
    points_cam = pred["pts3d_cam"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    colors = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    
    
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points_cam)
    pcd.colors = o3d.utility.Vector3dVector(colors)
    
    
    camera_pose = pred["camera_poses"].squeeze().cpu().numpy()
    
    
    pcd.transform(camera_pose)

    camera_center = camera_pose[:3, 3]
    camera_positions.append(camera_center)
    
    
    camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
    camera_frame.transform(camera_pose)
    
    
    geometries_to_draw.append(pcd)
    geometries_to_draw.append(camera_frame)

if len(camera_positions) > 1:
    # Define the points for the line set
    line_points = o3d.utility.Vector3dVector(camera_positions)
    # Define which points to connect (0->1, 1->2, etc.)
    line_indices = [[i, i + 1] for i in range(len(camera_positions) - 1)]
    lines = o3d.utility.Vector2iVector(line_indices)
    
    # Create the LineSet object
    camera_path = o3d.geometry.LineSet(points=line_points, lines=lines)
    
    # Set the color of the path to red
    camera_path.paint_uniform_color([1, 0, 0])
    
    # Add the path to our list of things to draw
    geometries_to_draw.append(camera_path)

transform_matrix = np.array([
    [1,  0,  0,  0],  # Stays the same
    [0, -1,  0,  0],  # Inverts Y
    [0,  0, -1,  0],  # Inverts Z
    [0,  0,  0,  1]
])

for geometry in geometries_to_draw:
    geometry.transform(transform_matrix)
# 4. Display all the geometries together in one window
print(f"Displaying combined scene with {len(predictions)} point clouds...")
o3d.visualization.draw_geometries(geometries_to_draw)

Displaying combined scene with 5 point clouds...


In [7]:
# Optional config for better memory efficiency
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Required imports
import torch
from mapanything.models import MapAnything
from mapanything.utils.image import load_images
import open3d as o3d
import numpy as np
import glob # To find all image files

# --- Initial Setup ---
device = "cuda" if torch.cuda.is_available() else "cpu"
model = MapAnything.from_pretrained("facebook/map-anything").to(device)

# --- 1. Keyframing and Batching Setup ---

# Directory containing your sequential PNG images
image_directory = "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/"

# Get a sorted list of all PNG files in the directory
all_image_files = sorted(glob.glob(os.path.join(image_directory, '*.png')))

# Select keyframes. A stride of 30 for 60fps video is like taking 2 frames per second.
# Adjust the stride based on how fast the camera is moving.
keyframe_stride = 120
keyframes = all_image_files[::keyframe_stride]

# Define the batch size based on your GPU limit
batch_size = 5

print(f"Found {len(all_image_files)} total images. Processing {len(keyframes)} keyframes in batches of {batch_size}.")

# --- 2. Process Batches and Accumulate the Map ---

# A list to hold all the 3D objects for the final map
geometries_to_draw = []
# The main transformation that chains all batches to the world origin
global_transform = np.identity(4)

# Add a coordinate frame for the world origin
world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0, origin=[0, 0, 0])
geometries_to_draw.append(world_frame)

# Process the keyframes in batches
for i in range(0, len(keyframes), batch_size):
    # Get the current batch of image paths
    batch_paths = keyframes[i:i + batch_size]
    if not batch_paths:
        continue
    
    print(f"Processing batch {i//batch_size + 1}...")
    
    # Load and run inference on the current batch
    views = load_images(batch_paths)
    predictions = model.infer(
        views,
        memory_efficient_inference=True, # Use this for larger datasets
        use_amp=True,
        amp_dtype="bf16",
        apply_mask=True,
        mask_edges=True
    )
    
    # Process the results of the current batch
    for pred in predictions:
        points_cam = pred["pts3d_cam"].squeeze().cpu().numpy().reshape(-1, 3)
        colors = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
        local_camera_pose = pred["camera_poses"].squeeze().cpu().numpy()

        # This is the key SLAM step: Transform the local pose into the global frame
        current_global_pose = global_transform @ local_camera_pose
        
        # Create and transform the point cloud
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(points_cam)
        pcd.colors = o3d.utility.Vector3dVector(colors)
        pcd.transform(current_global_pose)
        
        # Create and transform the camera frame visualization
        camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5)
        camera_frame.transform(current_global_pose)
        
        geometries_to_draw.append(pcd)
        geometries_to_draw.append(camera_frame)
    
    # Update the global transform to chain the next batch correctly
    # The new global transform is the pose of the LAST camera in the current batch
    global_transform = current_global_pose

# --- 3. Final Visualization ---

# (Optional) You can add the camera path and final transform here if desired

print("Displaying final accumulated map...")
o3d.visualization.draw_geometries(geometries_to_draw, width=1280, height=720)

Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /home/tong/.cache/torch/hub/facebookresearch_dinov2_main


Found 1860 total images. Processing 16 keyframes in batches of 5.
Processing batch 1...
Processing batch 2...
Processing batch 3...
Processing batch 4...
Displaying final accumulated map...


## remember the last camera pose ##

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import glob
import numpy as np
import open3d as o3d
import torch

from mapanything.models import MapAnything
from mapanything.utils.image import load_images

# --- Initial Setup ---
device = "cuda" if torch.cuda.is_available() else "cpu"
model = MapAnything.from_pretrained("facebook/map-anything").to(device)

# --- 1. Keyframing and Batching Setup ---
image_directory = (
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/"
    "Software/scrippsDivesWithDepth/depth3/linearPNG/"
)
all_image_files = sorted(glob.glob(os.path.join(image_directory, "*.png")))
keyframe_stride = 60
keyframes = all_image_files[::keyframe_stride]
batch_size = 3

print(f"Found {len(all_image_files)} frames → {len(keyframes)} keyframes.")

# --- 2. Process Batches and Accumulate the Map ---
geometries = []

world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0, origin=[0, 0, 0])
geometries.append(world_frame)

last_pose = None          # will stay shaped (1, 4, 4)
last_view = None          # full metadata dict from load_images

# --- Process FIRST BATCH Normally ---
print("Processing Batch 1...")
batch_1_paths = keyframes[0:batch_size]
views_batch_1 = load_images(batch_1_paths)
predictions_batch_1 = model.infer(views_batch_1, memory_efficient_inference=True, use_amp=True)

for i, pred in enumerate(predictions_batch_1):
    points_cam = pred["pts3d_cam"].squeeze().cpu().numpy().reshape(-1, 3)
    colors = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
    if colors.dtype != np.float64:
        colors = colors.astype(np.float64)
    if colors.max() > 1.0:
        colors /= 255.0

    global_pose = pred["camera_poses"].squeeze(0).cpu().numpy()

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points_cam)
    pcd.colors = o3d.utility.Vector3dVector(colors)
    pcd.transform(global_pose)

    camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5)
    camera_frame.transform(global_pose)

    geometries.append(pcd)
    geometries.append(camera_frame)

    if i == len(predictions_batch_1) - 1:
        last_pose = pred["camera_poses"].detach().cpu()       # keep batch dim
        last_view = views_batch_1[i]                          # store full dict

# --- Process SECOND BATCH using the Pose Prior ---
if len(keyframes) > batch_size:
    print("Processing Batch 2 (with pose prior)...")
    batch_2_paths = keyframes[batch_size : batch_size * 2]
    new_views_batch_2 = load_images(batch_2_paths)

    # # keep existing metadata and attach the pose
    # prior_view = dict(last_view)
    # prior_view["camera_poses"] = last_pose.clone()

    prior_view = {
        "img": last_view["img"],              # Image tensor from the last view
        "camera_poses": last_pose,        # The known pose tensor
        "data_norm_type": last_view["data_norm_type"]
    }

    views_with_prior = [prior_view]
    views_with_prior.extend(new_views_batch_2)

    predictions_batch_2 = model.infer(
        views_with_prior,
        memory_efficient_inference=True,
        use_amp=True,
    )

    for idx, pred in enumerate(predictions_batch_2):
        if idx == 0:  # skip the injected prior
            continue

        points_cam = pred["pts3d_cam"].squeeze().cpu().numpy().reshape(-1, 3)
        colors = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
        if colors.dtype != np.float64:
            colors = colors.astype(np.float64)
        if colors.max() > 1.0:
            colors /= 255.0

        global_pose = pred["camera_poses"].squeeze(0).cpu().numpy()

        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(points_cam)
        pcd.colors = o3d.utility.Vector3dVector(colors)
        pcd.transform(global_pose)

        camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5)
        camera_frame.transform(global_pose)

        geometries.append(pcd)
        geometries.append(camera_frame)

# --- 3. Final Visualization ---
print("Displaying final accumulated map...")
o3d.visualization.draw_geometries(geometries, width=1280, height=720)


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /home/tong/.cache/torch/hub/facebookresearch_dinov2_main


Found 1860 frames → 31 keyframes.
Processing Batch 1...
Processing Batch 2 (with pose prior)...


ValueError: View 0 missing required keys: {'data_norm_type'}

In [10]:
# Optional config for better memory efficiency
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Required imports
import torch
from mapanything.models import MapAnything
from mapanything.utils.image import load_images
import open3d as o3d
import numpy as np
import glob

# --- Initial Setup ---
device = "cuda" if torch.cuda.is_available() else "cpu"
model = MapAnything.from_pretrained("facebook/map-anything").to(device)

# --- 1. Batching Setup with Keyframing ---
image_directory = "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/Software/scrippsDivesWithDepth/depth3/linearPNG/"

# Get a sorted list of ALL PNG files
all_image_files = sorted(glob.glob(os.path.join(image_directory, '*.png')))


# A stride of 30 skips 29 frames, processing roughly 2 frames per second from 60fps footage.
keyframe_stride = 120
keyframes_to_process = all_image_files[::keyframe_stride]

# Define the batch size
batch_size = 5
num_batches = int(np.ceil(len(keyframes_to_process) / batch_size))

print(f"Found {len(all_image_files)} total images. Processing {len(keyframes_to_process)} keyframes in {num_batches} batches of {batch_size}.")

# --- 2. Process Batches and Accumulate the Map ---
geometries_to_draw = []
global_transform = np.identity(4)

world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0, origin=[0, 0, 0])
geometries_to_draw.append(world_frame)

# Process the selected keyframes in batches
for i in range(0, len(keyframes_to_process), batch_size):
    batch_paths = keyframes_to_process[i:i + batch_size]
    if not batch_paths:
        continue
    
    print(f"Processing batch {i//batch_size + 1}/{num_batches}...")
    
    views = load_images(batch_paths)
    predictions = model.infer(
        views,
        memory_efficient_inference=True,
        use_amp=True,
        amp_dtype="bf16",
        apply_mask=True,
        mask_edges=True
    )
    
    for pred in predictions:
        points_cam = pred["pts3d_cam"].squeeze().cpu().numpy().reshape(-1, 3)
        colors = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
        local_camera_pose = pred["camera_poses"].squeeze().cpu().numpy()

        # Chain the new pose to the end of the previous batch's pose
        current_global_pose = global_transform @ local_camera_pose
        
        # Create and transform the point cloud
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(points_cam)
        pcd.colors = o3d.utility.Vector3dVector(colors)
        pcd.transform(current_global_pose)
        
        # Create and transform the camera frame
        camera_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5)
        camera_frame.transform(current_global_pose)
        
        geometries_to_draw.append(pcd)
        geometries_to_draw.append(camera_frame)
    
    # Update the global transform to be the pose of the LAST camera in this batch
    global_transform = current_global_pose
    
    # Clean up memory for the next batch
    del predictions, views
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# --- 3. Final Visualization ---
print("Displaying final accumulated map...")
o3d.visualization.draw_geometries(geometries_to_draw, width=1280, height=720)

Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /home/tong/.cache/torch/hub/facebookresearch_dinov2_main


Found 1860 total images. Processing 16 keyframes in 4 batches of 5.
Processing batch 1/4...
Processing batch 2/4...
Processing batch 3/4...
Processing batch 4/4...
Displaying final accumulated map...


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import glob
import numpy as np
import open3d as o3d
import torch

from mapanything.models import MapAnything
from mapanything.utils.image import load_images


# def set_view_from_pose(view_ctl, pose, follow_distance=5.0):
#     """Align the Open3D camera with the given world pose."""
#     params = view_ctl.convert_to_pinhole_camera_parameters()
#     params.extrinsic = np.linalg.inv(pose)  # world → camera
#     view_ctl.convert_from_pinhole_camera_parameters(params, allow_arbitrary=True)

#     zoom = np.clip(1.0 / max(follow_distance, 1e-6), 0.02, 2.0)
#     view_ctl.set_zoom(zoom)

def set_view_from_pose(view_ctl, pose, follow_distance=5.0):
    params = view_ctl.convert_to_pinhole_camera_parameters()
    extrinsic = np.eye(4)
    extrinsic[:3, :3] = np.array(
        [[1.0, 0.0, 0.0],
         [0.0, 0.0, -1.0],
         [0.0, 1.0, 0.0]]
    )
    extrinsic[:3, 3] = np.array([0.0, 0.0, 10.0])
    params.extrinsic = extrinsic
    view_ctl.convert_from_pinhole_camera_parameters(params, allow_arbitrary=True)
    view_ctl.set_zoom(0.8)



device = "cuda" if torch.cuda.is_available() else "cpu"
model = MapAnything.from_pretrained("facebook/map-anything").to(device)

image_directory = (
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/"
    "Software/scrippsDivesWithDepth/depth3/linearPNG/"
)
all_image_files = sorted(glob.glob(os.path.join(image_directory, "*.png")))
keyframe_stride = 120
keyframes = all_image_files[::keyframe_stride]
batch_size = 5

print(f"Found {len(all_image_files)} images, processing {len(keyframes)} keyframes.")

vis = o3d.visualization.Visualizer()
vis.create_window("Live MapAnything Accumulation", width=1280, height=720)
render_opts = vis.get_render_option()
render_opts.background_color = np.array([0.02, 0.02, 0.02])
render_opts.point_size = 1.5

view_ctl = vis.get_view_control()

world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0)
vis.add_geometry(world_frame)

geometries_to_draw = [world_frame]
global_transform = np.eye(4)
follow_distance = 5.0

for i in range(0, len(keyframes), batch_size):
    batch_paths = keyframes[i : i + batch_size]
    if not batch_paths:
        continue

    print(f"Processing batch {i // batch_size + 1} / "
          f"{int(np.ceil(len(keyframes) / batch_size))}...")
    views = load_images(batch_paths)
    predictions = model.infer(
        views,
        memory_efficient_inference=True,
        use_amp=True,
        amp_dtype="bf16",
        apply_mask=True,
        mask_edges=True,
    )

    for pred in predictions:
        points_cam = pred["pts3d_cam"].squeeze().cpu().numpy().reshape(-1, 3)
        colors = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)

        if colors.dtype != np.float64:
            colors = colors.astype(np.float64)
        if colors.max() > 1.0:
            colors /= 255.0

        local_pose = pred["camera_poses"].squeeze().cpu().numpy()
        current_global_pose = global_transform @ local_pose

        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(points_cam)
        pcd.colors = o3d.utility.Vector3dVector(colors)
        pcd.transform(current_global_pose)

        cam_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5)
        cam_frame.transform(current_global_pose)

        vis.add_geometry(pcd)
        vis.add_geometry(cam_frame)
        geometries_to_draw.extend([pcd, cam_frame])

        set_view_from_pose(view_ctl, current_global_pose, follow_distance)
        vis.poll_events()
        vis.update_renderer()

    global_transform = current_global_pose  # chain to next batch

print("Accumulation complete — rotate/zoom freely, press Q to exit.")
vis.run()
vis.destroy_window()


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /home/tong/.cache/torch/hub/facebookresearch_dinov2_main


Found 1860 images, processing 372 keyframes.
Processing batch 1 / 75...
Processing batch 2 / 75...
Processing batch 3 / 75...
Processing batch 4 / 75...
Processing batch 5 / 75...
Processing batch 6 / 75...
Processing batch 7 / 75...
Processing batch 8 / 75...
Processing batch 9 / 75...
Processing batch 10 / 75...
Processing batch 11 / 75...
Processing batch 12 / 75...
Processing batch 13 / 75...
Processing batch 14 / 75...
Processing batch 15 / 75...
Processing batch 16 / 75...
Processing batch 17 / 75...
Processing batch 18 / 75...
Processing batch 19 / 75...
Processing batch 20 / 75...
Processing batch 21 / 75...
Processing batch 22 / 75...
Processing batch 23 / 75...


OutOfMemoryError: CUDA out of memory. Tried to allocate 308.00 MiB. GPU 0 has a total capacity of 7.58 GiB of which 380.75 MiB is free. Including non-PyTorch memory, this process has 6.36 GiB memory in use. Of the allocated memory 5.61 GiB is allocated by PyTorch, and 69.97 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import glob
import queue
import threading
import numpy as np
import open3d as o3d
import torch

from mapanything.models import MapAnything
from mapanything.utils.image import load_images

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
IMAGE_DIR = (
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/"
    "Software/scrippsDivesWithDepth/depth3/linearPNG/"
)
BATCH_SIZE = 1                         # drop load to avoid CUDA OOM
POINT_SIZE = 1.5
BACKGROUND = (0.02, 0.02, 0.02)
WORLD_FRAME_SIZE = 1.0
CAM_FRAME_SIZE = 0.5
QUEUE_BURST = 2                             # geometries added per UI tick

COORD_TRANSFORM = np.array(
    [
        [1.0, 0.0, 0.0, 0.0],
        [0.0, -1.0, 0.0, 0.0],
        [0.0, 0.0, -1.0, 0.0],
        [0.0, 0.0, 0.0, 1.0],
    ],
    dtype=np.float64,
)

SENTINEL = object()

# ---------------------------------------------------------------------------
# Load model and data
# ---------------------------------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
model = MapAnything.from_pretrained("facebook/map-anything").to(device)

all_frames = sorted(glob.glob(os.path.join(IMAGE_DIR, "*.png")))
num_batches = int(np.ceil(len(all_frames) / BATCH_SIZE))

print(f"Found {len(all_frames)} images; streaming {num_batches} batches of {BATCH_SIZE}.")

# ---------------------------------------------------------------------------
# Open3D setup
# ---------------------------------------------------------------------------
vis = o3d.visualization.VisualizerWithKeyCallback()
vis.create_window("MapAnything Live Accumulator", width=1280, height=720)
opts = vis.get_render_option()
opts.background_color = np.asarray(BACKGROUND)
opts.point_size = POINT_SIZE

view_ctl = vis.get_view_control()
view_ctl.set_front([0.0, 0.0, -1.0])
view_ctl.set_up([0.0, 1.0, 0.0])
view_ctl.set_lookat([0.0, 0.0, 0.0])
view_ctl.set_zoom(0.5)

world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=WORLD_FRAME_SIZE)
world_frame.transform(COORD_TRANSFORM)
vis.add_geometry(world_frame)

stream_queue = queue.Queue()
stream_finished = {"done": False}

def animation_callback(vis):
    processed = 0
    while processed < QUEUE_BURST:
        try:
            item = stream_queue.get_nowait()
        except queue.Empty:
            break

        if item is SENTINEL:
            stream_finished["done"] = True
            vis.register_animation_callback(None)
            return False

        pcd, cam_frame = item
        vis.add_geometry(pcd)
        vis.add_geometry(cam_frame)
        vis.update_renderer()
        processed += 1

    return False

vis.register_animation_callback(animation_callback)

# ---------------------------------------------------------------------------
# Inference worker
# ---------------------------------------------------------------------------
def inference_worker():
    last_global_pose = None

    with torch.inference_mode():
        for batch_idx in range(num_batches):
            batch_paths = all_frames[batch_idx * BATCH_SIZE : (batch_idx + 1) * BATCH_SIZE]
            if not batch_paths:
                continue

            print(f"Processing batch {batch_idx + 1}/{num_batches}…")
            views = load_images(batch_paths)
            predictions = model.infer(
                views,
                memory_efficient_inference=True,
                use_amp=True,
                amp_dtype="bf16",
                apply_mask=False,
                mask_edges=False,
            )

            batch_anchor_pose = None
            global_transform = np.eye(4)

            for pred in predictions:
                pts_cam = pred["pts3d_cam"].squeeze().detach().cpu().numpy().reshape(-1, 3)
                colors = pred["img_no_norm"].squeeze().detach().cpu().numpy().reshape(-1, 3)

                if colors.dtype != np.float64:
                    colors = colors.astype(np.float64)
                if colors.max() > 1.0:
                    colors /= 255.0

                local_pose = pred["camera_poses"].squeeze().detach().cpu().numpy()

                if batch_anchor_pose is None:
                    if last_global_pose is None:
                        global_transform = np.eye(4)
                    else:
                        global_transform = last_global_pose @ np.linalg.inv(local_pose)
                    batch_anchor_pose = local_pose

                current_global_pose = global_transform @ local_pose
                final_pose = COORD_TRANSFORM @ current_global_pose

                pcd = o3d.geometry.PointCloud()
                pcd.points = o3d.utility.Vector3dVector(pts_cam)
                pcd.colors = o3d.utility.Vector3dVector(colors)
                voxel_size = 0.05
                pcd = pcd.voxel_down_sample(voxel_size)
                pcd.transform(final_pose)

                cam_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=CAM_FRAME_SIZE)
                cam_frame.transform(final_pose)

                stream_queue.put((pcd, cam_frame))
                last_global_pose = current_global_pose
            # if stream_queue.qsize() > 50:
            #     import time
            #     time.sleep(0.1) # Pause for 100ms
            del predictions, views
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    stream_queue.put(SENTINEL)
    print("Streaming complete — rotate/zoom freely, press Q to exit.")

worker = threading.Thread(target=inference_worker, daemon=True)
worker.start()

vis.run()
vis.destroy_window()


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /home/tong/.cache/torch/hub/facebookresearch_dinov2_main


Found 1860 images; streaming 1860 batches of 1.
Processing batch 1/1860…
Processing batch 2/1860…


Processing batch 3/1860…
Processing batch 4/1860…
Processing batch 5/1860…
Processing batch 6/1860…
Processing batch 7/1860…
Processing batch 8/1860…
Processing batch 9/1860…
Processing batch 10/1860…
Processing batch 11/1860…


In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import glob
import numpy as np
import open3d as o3d
import torch

from mapanything.models import MapAnything
from mapanything.utils.image import load_images

# --- Configuration ---
IMAGE_DIR = (
    "/run/user/1000/gvfs/smb-share:server=unas-pro.local,share=dwe_nas/"
    "Software/scrippsDivesWithDepth/depth3/linearPNG/"
)
BATCH_SIZE = 5  # Adjust based on your VRAM (e.g., 3 for an RTX 2060)
COORD_TRANSFORM = np.array([
    [1.0, 0.0, 0.0, 0.0],
    [0.0, -1.0, 0.0, 0.0],
    [0.0, 0.0, -1.0, 0.0],
    [0.0, 0.0, 0.0, 1.0],
], dtype=np.float64)

# --- Load Model and Data ---
device = "cuda" if torch.cuda.is_available() else "cpu"
model = MapAnything.from_pretrained("facebook/map-anything").to(device)

all_frames = sorted(glob.glob(os.path.join(IMAGE_DIR, "*.png")))
num_batches = int(np.ceil(len(all_frames) / BATCH_SIZE))
print(f"Found {len(all_frames)} images; processing {num_batches} batches of {BATCH_SIZE}.")

# --- Process Batches and Accumulate Map ---
geometries_to_draw = []
global_transform = np.identity(4)  # "Remembers" the pose of the last frame

with torch.inference_mode():
    for batch_idx in range(num_batches):
        batch_paths = all_frames[batch_idx * BATCH_SIZE : (batch_idx + 1) * BATCH_SIZE]
        if not batch_paths:
            continue

        print(f"Processing batch {batch_idx + 1}/{num_batches}…")
        views = load_images(batch_paths)
        predictions = model.infer(
            views, memory_efficient_inference=True, use_amp=True
        )

        # Process the results for the current batch
        for pred in predictions:
            pts_cam = pred["pts3d_cam"].squeeze().cpu().numpy().reshape(-1, 3)
            colors = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
            local_pose = pred["camera_poses"].squeeze().cpu().numpy()

            # Chain the local pose to the end of the previous batch's pose
            current_global_pose = global_transform @ local_pose

            # Create and place the point cloud
            pcd = o3d.geometry.PointCloud()
            pcd.points = o3d.utility.Vector3dVector(pts_cam)
            pcd.colors = o3d.utility.Vector3dVector(colors)
            pcd.transform(current_global_pose)
            geometries_to_draw.append(pcd)

            # Create and place the camera frame
            cam_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5)
            cam_frame.transform(current_global_pose)
            geometries_to_draw.append(cam_frame)

        # Update the global transform to be the pose of the LAST camera in this batch
        global_transform = current_global_pose
        
        # Clean up memory
        del predictions, views
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# --- Final Transformation and Visualization ---
print("Applying final coordinate transformation...")
world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0)
geometries_to_draw.append(world_frame)

for geometry in geometries_to_draw:
    geometry.transform(COORD_TRANSFORM)

print("Displaying final accumulated map. Press Q to exit.")
o3d.visualization.draw_geometries(geometries_to_draw)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /home/tong/.cache/torch/hub/facebookresearch_dinov2_main


Found 1860 images; processing 372 batches of 5.
Processing batch 1/372…
Processing batch 2/372…
Processing batch 3/372…
Processing batch 4/372…
Processing batch 5/372…
Processing batch 6/372…
Processing batch 7/372…
Processing batch 8/372…
Processing batch 9/372…
Processing batch 10/372…
Processing batch 11/372…
Processing batch 12/372…
Processing batch 13/372…
Processing batch 14/372…
Processing batch 15/372…
Processing batch 16/372…
Processing batch 17/372…
Processing batch 18/372…
Processing batch 19/372…
Processing batch 20/372…
Processing batch 21/372…
Processing batch 22/372…
Processing batch 23/372…
Processing batch 24/372…
Processing batch 25/372…
Processing batch 26/372…
Processing batch 27/372…
Processing batch 28/372…
Processing batch 29/372…
Processing batch 30/372…
Processing batch 31/372…
Processing batch 32/372…
Processing batch 33/372…
Processing batch 34/372…
Processing batch 35/372…
Processing batch 36/372…
Processing batch 37/372…
Processing batch 38/372…
Processing 

KeyboardInterrupt: 